# Baseline Models

Goal: set the performance floor that any real model must beat.
The test set is split off first and left untouched until the final evaluation.
All model comparison uses 5-fold stratified cross-validation on the training data only.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import train_test_split
from src.data import load_clean

df = load_clean("../data/telco_churn.csv")
X = df.drop(columns="Churn")
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))

(5634, 19) (1409, 19)
0.2654 0.2654


In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dummy = DummyClassifier(strategy="most_frequent")

res = cross_validate(dummy, X_train, y_train, cv=cv,
                     scoring=["accuracy", "roc_auc", "recall"])

for name in ["test_accuracy", "test_roc_auc", "test_recall"]:
    print(name, res[name].mean().round(3))

test_accuracy 0.735
test_roc_auc 0.5
test_recall 0.0


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
    ],
    remainder="passthrough",  # SeniorCitizen is already 0/1
)

models = {
    "logreg": LogisticRegression(max_iter=1000),
    "logreg_balanced": LogisticRegression(max_iter=1000, class_weight="balanced"),
}

rows = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    res = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring=["accuracy", "roc_auc", "recall", "precision"],
    )
    rows[name] = {m: res[f"test_{m}"].mean() for m in ["accuracy", "roc_auc", "recall", "precision"]}

pd.DataFrame(rows).T.round(3)

,accuracy,roc_auc,recall,precision
logreg,0.802,0.846,0.544,0.653
logreg_balanced,0.749,0.846,0.803,0.518


## Results: baselines (5-fold stratified CV on training data)

| Model | Accuracy | ROC-AUC | Recall | Precision |
|---|---|---|---|---|
| Always predict "no churn" | 0.735 | 0.500 | 0.000 | n/a |
| Logistic regression | 0.802 | 0.846 | 0.544 | 0.653 |
| Logistic regression (balanced) | 0.749 | 0.846 | 0.803 | 0.518 |

**Takeaways**
- The dummy baseline reaches 73.5% accuracy while catching no churners, which is why accuracy is not the headline metric here.
- Both logistic models reach ROC-AUC 0.846, so class weighting changes the operating point (the precision/recall trade-off), not the model's ranking ability.
- The preferred threshold should come from the relative cost of a retention offer versus a lost customer. I'll revisit this after trying stronger models.

In [4]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

preprocess_dense = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), cat_cols),
    ],
    remainder="passthrough",
)

tree_models = {
    "random_forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
    "hist_gb": HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=200, max_depth=3, random_state=42
    ),
}

for name, clf in tree_models.items():
    pipe = Pipeline([("prep", preprocess_dense), ("clf", clf)])
    res = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring=["accuracy", "roc_auc", "recall", "precision"],
    )
    rows[name] = {m: res[f"test_{m}"].mean() for m in ["accuracy", "roc_auc", "recall", "precision"]}

pd.DataFrame(rows).T.round(3)

,accuracy,roc_auc,recall,precision
logreg,0.802,0.846,0.544,0.653
logreg_balanced,0.749,0.846,0.803,0.518
random_forest,0.803,0.844,0.502,0.673
hist_gb,0.805,0.848,0.522,0.672


In [5]:
from sklearn.model_selection import cross_val_score

candidates = {
    "logreg": Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))]),
    "random_forest": Pipeline([("prep", preprocess_dense), ("clf", tree_models["random_forest"])]),
    "hist_gb": Pipeline([("prep", preprocess_dense), ("clf", tree_models["hist_gb"])]),
}

fold_auc = {
    name: cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    for name, pipe in candidates.items()
}

for name, s in fold_auc.items():
    print(f"{name:14s} folds={s.round(3)}  mean={s.mean():.4f}  std={s.std(ddof=1):.4f}")

print()
for name in ["random_forest", "hist_gb"]:
    d = fold_auc[name] - fold_auc["logreg"]
    print(f"{name} minus logreg: {d.round(4)}  mean={d.mean():.4f}  std={d.std(ddof=1):.4f}")

logreg         folds=[0.848 0.825 0.842 0.863 0.853]  mean=0.8462  std=0.0141
random_forest  folds=[0.839 0.83  0.845 0.849 0.858]  mean=0.8442  std=0.0104
hist_gb        folds=[0.843 0.831 0.844 0.862 0.858]  mean=0.8477  std=0.0124

random_forest minus logreg: [-0.0091  0.0052  0.0031 -0.0143  0.0051]  mean=-0.0020  std=0.0091
hist_gb minus logreg: [-0.0046  0.0057  0.0025 -0.001   0.0048]  mean=0.0015  std=0.0043


In [6]:
from sklearn.model_selection import cross_val_predict
import numpy as np

final_pipe = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))])
oof_proba = cross_val_predict(final_pipe, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
y_arr = y_train.values

# ASSUMPTIONS (illustrative; replace with real business numbers)
offer_cost = 20        # cost of one retention offer
save_value = 200       # value of keeping a customer who would have churned
accept_rate = 0.30     # share of would-be churners who stay after an offer

rows_thr = []
for thr in np.arange(0.05, 0.81, 0.05):
    flagged = oof_proba >= thr
    tp = (flagged & (y_arr == 1)).sum()
    n_flag = flagged.sum()
    rows_thr.append({
        "threshold": round(thr, 2),
        "flagged": n_flag,
        "precision": tp / n_flag if n_flag else np.nan,
        "recall": tp / (y_arr == 1).sum(),
        "net_value": tp * accept_rate * save_value - n_flag * offer_cost,
    })

thr_table = pd.DataFrame(rows_thr).round(3)
print(thr_table.to_string(index=False))
print("\nBest threshold by net value:", thr_table.loc[thr_table.net_value.idxmax(), "threshold"])
print("Theoretical break-even probability:", round(offer_cost / (accept_rate * save_value), 3))

 threshold  flagged  precision  recall  net_value
      0.05     4112      0.354   0.974     5120.0
      0.10     3519      0.400   0.942    14100.0
      0.15     3120      0.438   0.914    19560.0
      0.20     2694      0.475   0.856    22920.0
      0.25     2397      0.509   0.817    25320.0
      0.30     2117      0.542   0.767    26480.0
      0.35     1880      0.570   0.717    26720.0
      0.40     1673      0.594   0.664    26120.0
      0.45     1443      0.624   0.603    25200.0
      0.50     1247      0.653   0.544    23900.0
      0.55     1060      0.686   0.486    22420.0
      0.60      835      0.710   0.397    18880.0
      0.65      605      0.752   0.304    15200.0
      0.70      405      0.780   0.211    10860.0
      0.75      176      0.847   0.100     5420.0
      0.80       48      0.938   0.030     1740.0

Best threshold by net value: 0.35
Theoretical break-even probability: 0.333


## Model comparison (paired 5-fold ROC-AUC)

| Model | Mean AUC | Std across folds |
|---|---|---|
| Logistic regression | 0.8462 | 0.0141 |
| Random forest | 0.8442 | 0.0104 |
| Gradient boosting | 0.8477 | 0.0124 |

Paired differences against logistic regression flip sign across folds (random forest: -0.002 ± 0.009; gradient boosting: +0.0015 ± 0.004), and the fold-to-fold noise is larger than any gap. The models are statistically indistinguishable on this data, so I'm choosing logistic regression for its simplicity and interpretability.

In [7]:
from sklearn.metrics import roc_auc_score, confusion_matrix

THRESHOLD = 0.35  # chosen on training data only (out-of-fold net value)

final_pipe.fit(X_train, y_train)
test_proba = final_pipe.predict_proba(X_test)[:, 1]
y_te = y_test.values
flagged = test_proba >= THRESHOLD

tn, fp, fn, tp = confusion_matrix(y_te, flagged).ravel()
precision = tp / (tp + fp)
recall = tp / (tp + fn)
auc = roc_auc_score(y_te, test_proba)

# Bootstrap 95% CI for ROC-AUC (resample test customers with replacement)
rng = np.random.default_rng(42)
boot = []
for _ in range(2000):
    idx = rng.integers(0, len(y_te), len(y_te))
    boot.append(roc_auc_score(y_te[idx], test_proba[idx]))
lo, hi = np.percentile(boot, [2.5, 97.5])

net_value = tp * accept_rate * save_value - (tp + fp) * offer_cost

print(f"ROC-AUC: {auc:.3f}  (95% CI {lo:.3f} to {hi:.3f})")
print(f"Flagged: {tp + fp} of {len(y_te)} customers ({(tp + fp) / len(y_te):.1%})")
print(f"Precision: {precision:.3f}   Recall: {recall:.3f}")
print(f"Confusion matrix: TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Net value on test set (placeholder assumptions): {net_value:,.0f}")

ROC-AUC: 0.842  (95% CI 0.818 to 0.863)
Flagged: 485 of 1409 customers (34.4%)
Precision: 0.544   Recall: 0.706
Confusion matrix: TN=814 FP=221 FN=110 TP=264
Net value on test set (placeholder assumptions): 6,140


In [8]:
prep = final_pipe.named_steps["prep"]
clf = final_pipe.named_steps["clf"]

coefs = pd.Series(clf.coef_[0], index=prep.get_feature_names_out())
coef_df = pd.DataFrame({"coef": coefs, "odds_ratio": np.exp(coefs)}).round(3)
coef_df = coef_df.reindex(coef_df["coef"].abs().sort_values(ascending=False).index)

print(coef_df.head(15))

                                            coef  odds_ratio
num__tenure                               -1.244       0.288
cat__Contract_Two year                    -0.815       0.442
cat__InternetService_DSL                  -0.701       0.496
num__MonthlyCharges                       -0.600       0.549
cat__InternetService_Fiber optic           0.594       1.811
cat__Contract_Month-to-month               0.529       1.698
num__TotalCharges                          0.520       1.682
cat__PaperlessBilling_Yes                  0.370       1.448
cat__StreamingTV_No internet service      -0.349       0.705
cat__TechSupport_No internet service      -0.349       0.705
cat__OnlineBackup_No internet service     -0.349       0.705
cat__StreamingMovies_No internet service  -0.349       0.705
cat__OnlineSecurity_No internet service   -0.349       0.705
cat__DeviceProtection_No internet service -0.349       0.705
cat__InternetService_No                   -0.349       0.705


## Final evaluation (held-out test set, used once)

Model: logistic regression, decision threshold 0.35 (chosen on out-of-fold training predictions from an explicit cost assumption).

| Metric | Value |
|---|---|
| ROC-AUC | 0.842 (95% bootstrap CI 0.818 to 0.863) |
| Precision | 0.544 |
| Recall | 0.706 |
| Customers flagged | 485 of 1,409 (34.4%) |

The test AUC is consistent with the cross-validation estimate (0.846), so there is no sign of overfitting.
The net-value figures depend on placeholder assumptions (offer cost 20, save value 200, accept rate 30%) and should be recomputed with real business numbers.

## What the model learned (logistic regression coefficients)

**Consistent with the EDA, and robust to the overlap between features:**
- Longer tenure strongly reduces churn risk (odds about 0.29x per standard deviation).
- Two-year contracts reduce risk; month-to-month contracts increase it.
- Fiber optic customers are at higher risk than DSL customers; customers with no internet service are at much lower risk.
- Paperless billing is associated with higher risk.

**Interpretation caveats:**
- `tenure`, `MonthlyCharges`, and `TotalCharges` overlap heavily, so the signs on `MonthlyCharges` (negative) and `TotalCharges` (positive) should not be read as pricing effects.
- The "No internet service" indicator columns are perfectly redundant. Regularization splits their weight equally, so the seven identical coefficients represent one combined effect (about -2.4 on the log-odds scale).
- These are associations in observational data, not causal effects.